# Feature Attribution using PageRank

In [1]:
import sys

!{sys.executable} -m pip install nbimporter

In [2]:
import numpy as np
import pandas as pd
import zipfile as zf
import nbimporter
import sklearn.ensemble

import Ft_Att_Rank as far
import GraphPR as gpr
import PageRank as pr

In [3]:
# https://www.kaggle.com/c/titanic
ds= zf.ZipFile('datasets/titanic.zip')

train_data= pd.read_csv(ds.open('train.csv'))
test_data= pd.read_csv(ds.open('test.csv'))

X_all= pd.concat([train_data[['PassengerId','Pclass','Sex','Age','SibSp','Parch','Fare','Embarked']],
                   test_data[['PassengerId','Pclass','Sex','Age','SibSp','Parch','Fare','Embarked']]]).set_index('PassengerId')

y_train= train_data[['PassengerId','Survived']].set_index('PassengerId')['Survived']

numeric_columns= ['Age','SibSp','Parch','Fare']
categor_columns= list(filter(lambda x:x not in numeric_columns,X_all.columns))

X_train= X_all.iloc[:len(train_data),:].copy()
X_test= X_all.iloc[len(train_data):].copy()

In [4]:
X_train= far.pre_proc_fillna_num_fts(X_train,numeric_columns,num_type='median')
X_train= far.pre_proc_fillna_cat_fts(X_train,categor_columns,cat_type='mode')

X_train_ohe= pd.get_dummies(X_train,columns=categor_columns)

X_train= far.normalize_selected_cols(X_train, numeric_columns)
X_train_ohe= far.normalize_selected_cols(X_train_ohe, numeric_columns)

X_train.shape

(891, 7)

In [5]:
X_to_split= X_train_ohe.copy()
Y_to_split= y_train.copy()

In [6]:
target_index= 1

target_instance= X_to_split.loc[X_to_split.index== target_index]
target_label= Y_to_split.loc[Y_to_split.index== target_index]

X_no_targt= X_to_split.drop(target_instance.index)
Y_no_targt= Y_to_split.drop(index= target_index)

In [7]:
# ML model setup

rf= sklearn.ensemble.RandomForestClassifier(n_estimators=500,n_jobs=2)

In [8]:
mean_acc_all, acc_no_i, acc_no_ij= far.kfoldnn_remove_and_retrain(rf, X_no_targt, Y_no_targt, target_instance, 
                                                              target_label, numeric_columns, num_type='mean', 
                                                              cat_type='median', test_size= 0.2)

In [9]:
print(mean_acc_all)
print("-----------------------------")
#print(acc_no_i)
print(np.around(acc_no_i, decimals=3))
print("-----------------------------")
#print(acc_no_ij)
print(np.around(acc_no_ij, decimals=3))

0.855944055944056
-----------------------------
[0.88  0.866 0.859 0.376 0.871 0.857 0.871 0.856 0.856 0.867 0.867 0.871]
-----------------------------
[[0.    0.883 0.836 0.874 0.88  0.877 0.877 0.88  0.878 0.88  0.876 0.877]
 [0.878 0.    0.878 0.436 0.866 0.864 0.878 0.864 0.86  0.862 0.876 0.876]
 [0.838 0.871 0.    0.467 0.856 0.871 0.874 0.859 0.857 0.848 0.86  0.874]
 [0.87  0.436 0.46  0.    0.379 0.471 0.473 0.368 0.464 0.386 0.494 0.501]
 [0.877 0.864 0.856 0.378 0.    0.859 0.883 0.863 0.859 0.871 0.88  0.874]
 [0.878 0.862 0.863 0.476 0.864 0.    0.873 0.862 0.873 0.86  0.871 0.871]
 [0.877 0.874 0.871 0.42  0.883 0.873 0.    0.871 0.878 0.87  0.878 0.877]
 [0.878 0.878 0.862 0.383 0.876 0.876 0.878 0.    0.663 0.852 0.883 0.877]
 [0.877 0.877 0.874 0.376 0.863 0.862 0.863 0.669 0.    0.87  0.883 0.876]
 [0.877 0.871 0.87  0.378 0.855 0.857 0.871 0.869 0.869 0.    0.857 0.782]
 [0.876 0.866 0.856 0.492 0.878 0.874 0.877 0.878 0.867 0.874 0.    0.876]
 [0.878 0.876 0.874 0.4

In [11]:
num_fts= acc_no_ij.shape[0]

# get the probability matrix
p_matrix1= far.get_p_matrix_v1(num_fts, acc_no_i, acc_no_ij)
p_matrix2= far.get_p_matrix_v2(num_fts, acc_no_i, acc_no_ij)
p_matrix3= far.get_p_matrix_v3(num_fts, mean_acc_all, acc_no_i, acc_no_i, acc_no_ij)
p_matrix4= far.get_p_matrix_v4(num_fts, mean_acc_all, acc_no_i, acc_no_i, acc_no_ij)

# finding the stationary distribution
st_matrix1= np.asarray(p_matrix1)
st_matrix2= np.asarray(p_matrix2)
st_matrix3= np.asarray(p_matrix3)
st_matrix4= np.asarray(p_matrix4)

# now convert to right stochastic matrix - a real square matrix, with each row summing to 1
st_matrix1= far.to_row_stochastic_matrix(st_matrix1)
st_matrix2= far.to_row_stochastic_matrix(st_matrix2)
st_matrix3= far.to_row_stochastic_matrix(st_matrix3)
st_matrix4= far.to_row_stochastic_matrix(st_matrix4)

In [62]:
import networkx as nx

def run_lib_pr(graph_matrix, ft_names):
    
    num_fts= graph_matrix.shape[0]
    
    D= nx.DiGraph()

    for i in range(num_fts):
        for j in range(num_fts):
            if (i!= j):
                D.add_weighted_edges_from([(ft_names[i],ft_names[j],graph_matrix[i,j])])
                
    pRank= pd.Series(nx.pagerank(D, max_iter=100, alpha=0.85, tol=1.0e-6))

    return pRank.sort_values(ascending=False)

In [63]:
run_lib_pr(st_matrix1, X_train_ohe.columns)

Fare          0.354597
Age           0.149903
Parch         0.122303
Embarked_S    0.060973
Embarked_Q    0.056398
Pclass_3      0.050864
Pclass_2      0.047944
Sex_male      0.046006
SibSp         0.039120
Embarked_C    0.027620
Sex_female    0.026221
Pclass_1      0.018051
dtype: float64

In [65]:
# using my PR to compare results
def run_my_pr(graph_matrix, ft_names):
    
    file_path= 'datasets/FAR_data.txt'
    
    num_fts= graph_matrix.shape[0]

    f= open(file_path, 'w')

    for i in range(num_fts):
        for j in range(num_fts):
            line= (str(i) + ',' + str(j) + ',' + str(graph_matrix[i,j]) + '\n')
            f.write(line)

    f.close()

    graph= gpr.init_graph(file_path)

    myPRank= pd.Series(pr.run_PageRank(graph, iteration= 100, damping_factor= 0.95, tolerance= 1.0e-6))
    myPRank= myPRank.sort_values(ascending=False)

    ids_names= ft_names[myPRank.index]

    myPRank.index= ids_names

    return myPRank

In [66]:
run_my_pr(st_matrix1, X_train_ohe.columns)

Fare          0.304923
Age           0.160533
Parch         0.074396
SibSp         0.062853
Sex_female    0.054675
Embarked_C    0.051053
Pclass_3      0.049807
Pclass_2      0.049655
Pclass_1      0.048714
Embarked_S    0.048093
Embarked_Q    0.047945
Sex_male      0.047352
dtype: float64

In [54]:
run_lib_pr(st_matrix2, X_train_ohe.columns)

Fare          0.184380
Sex_female    0.152141
Sex_male      0.144291
Embarked_S    0.071810
Embarked_C    0.070742
Embarked_Q    0.067620
Parch         0.059208
Pclass_2      0.058610
Pclass_1      0.056754
Pclass_3      0.051405
SibSp         0.050905
Age           0.032135
dtype: float64

In [57]:
run_my_pr(st_matrix2, X_train_ohe.columns)

Fare          0.177912
Sex_female    0.115074
Parch         0.089445
SibSp         0.087180
Age           0.076876
Pclass_2      0.073446
Pclass_1      0.071612
Sex_male      0.069942
Embarked_C    0.068376
Pclass_3      0.059486
Embarked_Q    0.055903
Embarked_S    0.054747
dtype: float64

In [55]:
run_lib_pr(st_matrix3, X_train_ohe.columns)

Fare          0.160618
Sex_female    0.149773
Sex_male      0.145958
Embarked_C    0.068164
Embarked_Q    0.065526
Embarked_S    0.065377
Pclass_2      0.063919
Parch         0.062497
Pclass_1      0.058486
SibSp         0.057747
Pclass_3      0.052865
Age           0.049071
dtype: float64

In [58]:
run_my_pr(st_matrix3, X_train_ohe.columns)

Fare          0.154502
Sex_female    0.112722
Age           0.093888
SibSp         0.092370
Parch         0.090743
Pclass_2      0.075104
Pclass_1      0.073328
Sex_male      0.070223
Embarked_C    0.065092
Pclass_3      0.062961
Embarked_Q    0.055658
Embarked_S    0.053409
dtype: float64

In [56]:
run_lib_pr(st_matrix4, X_train_ohe.columns)

Fare          0.411741
Age           0.164954
Embarked_S    0.063377
Embarked_Q    0.056724
Pclass_3      0.054243
Parch         0.049322
Pclass_2      0.042142
Sex_male      0.040572
SibSp         0.039249
Embarked_C    0.029425
Pclass_1      0.027637
Sex_female    0.020616
dtype: float64

In [59]:
run_my_pr(st_matrix4, X_train_ohe.columns)

Fare          0.317876
Age           0.163654
Parch         0.065259
SibSp         0.064179
Pclass_3      0.050750
Pclass_1      0.050563
Sex_female    0.049687
Embarked_C    0.048683
Embarked_S    0.047861
Embarked_Q    0.047698
Pclass_2      0.047610
Sex_male      0.046180
dtype: float64

In [8]:
target_index= 1

target_instance= X_to_split.loc[X_to_split.index== target_index]
target_label= Y_to_split.loc[Y_to_split.index== target_index]

X_no_targt= X_to_split.drop(target_instance.index)
Y_no_targt= Y_to_split.drop(index= target_index)

In [9]:
train, test, labels_train, labels_test= far.knn_train_test_split(X_no_targt,Y_no_targt,
                                                                target_instance,target_label,test_size=0.5)

In [16]:
# here we'll only consider the test set from knn split because this set contains the test_size percentage 
# of nearest elements from the target instance
# target_X and target_Y is into test_X and test_Y, we drop the target instance again

new_X_no_tgt= test.drop(target_instance.index)
new_Y_no_tgt= labels_test.drop(index= target_index)


In [17]:
import time

start= time.time()


train_1, test_1, labels_train_1, labels_test_1= far.knn_train_test_split(new_X_no_tgt,new_Y_no_tgt,
                                                                    target_instance,target_label,test_size=0.2)

rf= sklearn.ensemble.RandomForestClassifier(n_estimators=500,n_jobs=2)


# Feature Attribution using Raking - Knn-Local - Remove and Retrain v1.0
repeat_train= 1
num_fts= len(train.columns)

acc_all_fts= far.train_model_get_acc_mean(rf, train_1, test_1, labels_train_1.values.ravel(), 
                                          labels_test_1.values.ravel(), repeat_train)

replace_ft= far.replace_values(train_1,train_1.columns,num_type='mean')

acc_no_i, acc_no_ij= far.remove_and_retrain_v1(rf, replace_ft, train_1, test_1, labels_train_1,
                                               labels_test_1, repeat_train)


end= time.time()
print("--- %s seconds ---" % np.round((end- start), 2))

--- 124.64 seconds ---


In [19]:
print(acc_all_fts)
print("-----------------------------")
#print(acc_no_i)
print(np.around(acc_no_i, decimals=3))
print("-----------------------------")
#print(acc_no_ij)
print(np.around(acc_no_ij, decimals=3))

0.8777777777777778
-----------------------------
[0.878 0.9   0.911 0.6   0.878 0.878 0.889 0.856 0.878 0.867 0.911 0.9  ]
-----------------------------
[[0.    0.911 0.811 0.922 0.878 0.878 0.878 0.878 0.878 0.878 0.878 0.878]
 [0.911 0.    0.889 0.522 0.867 0.9   0.867 0.844 0.822 0.889 0.889 0.889]
 [0.811 0.9   0.    0.6   0.889 0.878 0.911 0.867 0.878 0.878 0.911 0.9  ]
 [0.922 0.656 0.6   0.    0.6   0.722 0.6   0.6   0.6   0.6   0.722 0.722]
 [0.878 0.889 0.878 0.6   0.    0.878 0.867 0.889 0.844 0.878 0.9   0.911]
 [0.878 0.889 0.878 0.722 0.878 0.    0.911 0.878 0.878 0.9   0.9   0.9  ]
 [0.878 0.878 0.9   0.722 0.9   0.922 0.    0.867 0.867 0.867 0.9   0.9  ]
 [0.878 0.867 0.878 0.6   0.867 0.9   0.856 0.    0.567 0.822 0.878 0.9  ]
 [0.878 0.867 0.867 0.6   0.867 0.867 0.833 0.578 0.    0.856 0.9   0.9  ]
 [0.878 0.867 0.889 0.6   0.878 0.9   0.9   0.878 0.889 0.    0.9   0.878]
 [0.878 0.867 0.9   0.6   0.9   0.911 0.9   0.867 0.878 0.9   0.    0.922]
 [0.878 0.889 0.9   0.

In [21]:
start= time.time()


# Feature Attribution using Raking - Knn-Local - Kfold and knn Remove and Retrain
mean_acc_all, acc_no_i_k, acc_no_ij_k= far.kfoldnn_remove_and_retrain(rf, new_X_no_tgt,new_Y_no_tgt,
                                                                target_instance,target_label, numeric_columns, 
                                                                num_type='mean', cat_type='median', 
                                                                test_size= 0.2)

end= time.time()
print("--- %s seconds ---" % np.round((end- start), 2))

--- 594.55 seconds ---


In [22]:
print(mean_acc_all)
print("-----------------------------")
#print(acc_no_i)
print(np.around(acc_no_i_k, decimals=3))
print("-----------------------------")
#print(acc_no_ij)
print(np.around(acc_no_ij_k, decimals=3))

0.8861111111111111
-----------------------------
[0.869 0.897 0.892 0.617 0.889 0.892 0.894 0.892 0.883 0.872 0.892 0.892]
-----------------------------
[[0.    0.897 0.819 0.914 0.886 0.878 0.878 0.864 0.878 0.878 0.886 0.878]
 [0.892 0.    0.906 0.661 0.903 0.897 0.9   0.897 0.897 0.903 0.908 0.906]
 [0.806 0.906 0.    0.617 0.886 0.883 0.892 0.881 0.883 0.886 0.892 0.892]
 [0.914 0.689 0.667 0.    0.617 0.692 0.667 0.617 0.617 0.617 0.694 0.661]
 [0.864 0.903 0.889 0.617 0.    0.886 0.869 0.886 0.886 0.869 0.894 0.892]
 [0.886 0.9   0.889 0.664 0.892 0.    0.889 0.878 0.881 0.881 0.894 0.906]
 [0.864 0.9   0.892 0.639 0.881 0.908 0.    0.886 0.875 0.889 0.894 0.894]
 [0.886 0.903 0.883 0.611 0.872 0.886 0.883 0.    0.592 0.883 0.889 0.889]
 [0.878 0.903 0.878 0.617 0.889 0.886 0.878 0.619 0.    0.875 0.886 0.892]
 [0.869 0.906 0.892 0.656 0.886 0.881 0.883 0.878 0.883 0.    0.881 0.839]
 [0.878 0.9   0.894 0.667 0.897 0.894 0.889 0.872 0.886 0.889 0.    0.914]
 [0.886 0.908 0.889 0.